# 03 — Vision Transformers (ViT)

## What
Vision Transformer (Dosovitskiy et al. 2020) applies the transformer architecture directly to images by splitting them into fixed-size patches, linearly embedding each patch, and processing the sequence with standard transformer self-attention. No convolutions needed.

## Why
ViT demonstrated that transformers can match or exceed CNNs on image classification when trained at scale. More importantly, ViTs produce patch-level features that naturally integrate with text tokens in multimodal models — a key reason why CLIP, BLIP, and LLaVA all use ViT as the vision encoder.

## Community context
DeiT (Touvron et al.) made ViT trainable without massive datasets via knowledge distillation. Swin Transformer introduced hierarchical attention with local windows for dense prediction tasks. BEiT, MAE (Masked Autoencoder, He et al.), and DINOv2 developed self-supervised ViT pretraining — DINOv2 features in particular are used as the vision encoder in many recent VLMs.

In [ ]:
# Minimal ViT forward pass
import numpy as np

def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

class PatchEmbedding:
    def __init__(self, image_size=32, patch_size=8, n_channels=3, embed_dim=64):
        self.patch_size = patch_size
        self.n_patches = (image_size // patch_size) ** 2
        # Linear projection: (patch_size^2 * channels) -> embed_dim
        fan_in = patch_size * patch_size * n_channels
        self.W = np.random.randn(fan_in, embed_dim) * np.sqrt(2/fan_in)

    def __call__(self, image):
        """image: (H, W, C) -> (N_patches, embed_dim)"""
        H, W, C = image.shape
        P = self.patch_size
        patches = image.reshape(H//P, P, W//P, P, C)
        patches = patches.transpose(0,2,1,3,4).reshape(-1, P*P*C)  # (N, P*P*C)
        return patches @ self.W   # (N, embed_dim)

class MultiHeadAttention:
    def __init__(self, embed_dim, n_heads):
        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads
        scale = np.sqrt(embed_dim)
        self.Wq = np.random.randn(embed_dim, embed_dim) / scale
        self.Wk = np.random.randn(embed_dim, embed_dim) / scale
        self.Wv = np.random.randn(embed_dim, embed_dim) / scale
        self.Wo = np.random.randn(embed_dim, embed_dim) / scale

    def __call__(self, x):
        N, D = x.shape
        Q = x @ self.Wq  # (N, D)
        K = x @ self.Wk
        V = x @ self.Wv
        # Reshape to heads
        H, Dh = self.n_heads, self.head_dim
        Q = Q.reshape(N, H, Dh).transpose(1,0,2)  # (H, N, Dh)
        K = K.reshape(N, H, Dh).transpose(1,0,2)
        V = V.reshape(N, H, Dh).transpose(1,0,2)
        attn = softmax(Q @ K.transpose(0,2,1) / np.sqrt(Dh))  # (H, N, N)
        out  = (attn @ V).transpose(1,0,2).reshape(N, D)      # (N, D)
        return out @ self.Wo, attn

np.random.seed(42)
embed_dim = 64
patch_embed = PatchEmbedding(image_size=32, patch_size=8, n_channels=3, embed_dim=embed_dim)
mha = MultiHeadAttention(embed_dim=embed_dim, n_heads=4)

# Fake image
image = np.random.randn(32, 32, 3)
patch_tokens = patch_embed(image)                    # (16, 64)
cls_token = np.random.randn(1, embed_dim)           # (1, 64)
tokens = np.vstack([cls_token, patch_tokens])        # (17, 64)

# Add positional embedding (learned, here random)
pos_embed = np.random.randn(*tokens.shape) * 0.02
tokens = tokens + pos_embed

output, attn_weights = mha(tokens)
print(f'Input:  {tokens.shape}    (1 CLS + 16 patch tokens)')
print(f'Output: {output.shape}    (same shape, enriched with global context)')
print(f'Attention weights: {attn_weights.shape}  (4 heads, 17x17)')
print(f'\nCLS token output: {output[0, :8].round(3)}...')
print('CLS token aggregates info from all 16 patches for classification')

## Masked Autoencoder (MAE) Pretraining

MAE (He et al., 2021) achieves strong ViT representations by masking ~75% of patches and training the model to reconstruct them. The high mask ratio forces the encoder to learn rich semantic representations rather than simple pixel statistics.

In [ ]:
# MAE masking strategy
def mae_masking(n_patches, mask_ratio=0.75, seed=None):
    rng = np.random.default_rng(seed)
    n_masked = int(n_patches * mask_ratio)
    indices = rng.permutation(n_patches)
    masked_idx   = indices[:n_masked]
    unmasked_idx = indices[n_masked:]
    return sorted(unmasked_idx), sorted(masked_idx)

n_patches = 196  # 14x14 for 224px image with 16px patches
visible, masked = mae_masking(n_patches, mask_ratio=0.75, seed=42)
print(f'Total patches: {n_patches}')
print(f'Visible (encoder input): {len(visible)} patches ({100*len(visible)//n_patches}%)')
print(f'Masked (decoder target): {len(masked)} patches ({100*len(masked)//n_patches}%)')
print(f'\nFirst 10 visible patch indices: {visible[:10]}')
print(f'First 10 masked patch indices:  {masked[:10]}')
print('\nEncoder only processes visible patches -> 3x speedup vs full attention')